# 2. Предобработка

**Нужны:** все файлы из тетрадки 1 + `mentees.csv`, `mentors.csv`

**Создаёт:** `mentees_processed.csv`, `mentors_processed.csv`, `skill_vocabulary.csv`

In [1]:
import csv, math, os
from collections import Counter

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

mentees_raw = load_csv(os.path.join(BASE_DIR,"mentees.csv"))
mentors_raw = load_csv(os.path.join(BASE_DIR,"mentors.csv"))

domains = [r["domain_name"].strip()
           for r in load_csv(os.path.join(BASE_DIR,"ontology_domains.csv"))]

mapping = {r["profession"].strip(): {d: float(r.get(d,0)) for d in domains}
           for r in load_csv(os.path.join(BASE_DIR,"ontology_role_domain_mapping.csv"))}

LEVEL_SCORE = {"junior":1,"middle":2,"senior":3,"lead":4}

def norm_skills(raw):
    return [s.strip().lower() for s in (raw or "").split(";") if s.strip()]

def encode_set(raw):
    if not raw or not raw.strip(): return None
    s = raw.strip().lower()
    return set(s.split("+")) if "+" in s else {"online","offline","hybrid"} if s=="hybrid" else {s}

def domain_vec(prof):
    if prof in mapping:
        return [mapping[prof].get(d,0) for d in domains], False
    return [round(1/len(domains),4)]*len(domains), True

def vstr(v): return ",".join(str(x) for x in v) if v else ""
def sstr(s): return "|".join(sorted(s)) if s else ""

print(f"Загружено: {len(mentees_raw)} менти, {len(mentors_raw)} менторов")
print(f"Профессий в маппинге: {len(mapping)}")


Загружено: 5000 менти, 2000 менторов
Профессий в маппинге: 70


In [ ]:
processed_mentees = []
for r in mentees_raw:
    skills = norm_skills(r.get("skills",""))
    dv, df = domain_vec(r.get("profession","").strip())
    processed_mentees.append({
        "id":r["id"],"name":r["name"],
        "level_raw":r.get("level",""),
        "level_score":LEVEL_SCORE.get((r.get("level","") or "").lower(),1),
        "domain":r.get("domain",""),"profession":r.get("profession",""),
        "language":r.get("language",""),"language_set":sstr(encode_set(r.get("language",""))),
        "format":r.get("format",""),"format_set":sstr(encode_set(r.get("format",""))),
        "skills_normalized":"; ".join(skills),
        "skills_missing":"1" if not skills else "0",
        "domain_vector":vstr(dv),"domain_fallback":"1" if df else "0",
    })

processed_mentors = []
for r in mentors_raw:
    skills = norm_skills(r.get("skills",""))
    dv, df = domain_vec(r.get("profession","").strip())
    exp = int(r.get("experience_years",0) or 0)
    processed_mentors.append({
        "id":r["id"],"name":r["name"],
        "level_raw":r.get("level",""),
        "level_score":LEVEL_SCORE.get((r.get("level","") or "").lower(),2),
        "domain":r.get("domain",""),"profession":r.get("profession",""),
        "language":r.get("language",""),"language_set":sstr(encode_set(r.get("language",""))),
        "format":r.get("format",""),"format_set":sstr(encode_set(r.get("format",""))),
        "skills_normalized":"; ".join(skills),
        "skills_missing":"1" if not skills else "0",
        "domain_vector":vstr(dv),"domain_fallback":"1" if df else "0",
        "experience_years":exp,
        "experience_norm":round(math.log(exp+1)/math.log(21),4),
        "available":str(str(r.get("available","True")).lower() in ("true","1")),
        "hours_per_week":r.get("hours_per_week",""),
        "boosted":r.get("boosted","False"),"boost_k":float(r.get("boost_k",0) or 0),
    })

all_skills = sorted({s for m in processed_mentees+processed_mentors
                     for s in m["skills_normalized"].split("; ") if s})

MENTEE_F = ["id","name","level_raw","level_score","domain","profession",
            "language","language_set","format","format_set",
            "skills_normalized","skills_missing","domain_vector","domain_fallback"]
MENTOR_F = ["id","name","level_raw","level_score","domain","profession",
            "language","language_set","format","format_set",
            "skills_normalized","skills_missing","domain_vector","domain_fallback",
            "experience_years","experience_norm","available","hours_per_week",
            "boosted","boost_k"]

def save_csv(data, fields, path):
    with open(path,"w",newline="",encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        [w.writerow({k: row.get(k,"") for k in fields}) for row in data]

save_csv(processed_mentees, MENTEE_F, os.path.join(BASE_DIR,"mentees_processed.csv"))
save_csv(processed_mentors, MENTOR_F, os.path.join(BASE_DIR,"mentors_processed.csv"))

with open(os.path.join(BASE_DIR,"skill_vocabulary.csv"),"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["index","skill"])
    w.writeheader()
    [w.writerow({"index":i,"skill":s}) for i,s in enumerate(all_skills)]

n_sm = sum(1 for m in processed_mentees if m["skills_missing"]=="1")
n_mm = sum(1 for m in processed_mentors if m["skills_missing"]=="1")
print(f"mentees_processed.csv  ({len(processed_mentees)} строк, {n_sm} без навыков)")
print(f"mentors_processed.csv  ({len(processed_mentors)} строк, {n_mm} без навыков)")
print(f"skill_vocabulary.csv   ({len(all_skills)} навыков)")
print("Тетрадка 2 завершена!")


mentees_processed.csv  (5000 строк, 1001 без навыков)
mentors_processed.csv  (2000 строк, 307 без навыков)
skill_vocabulary.csv   (105 навыков)
Тетрадка 2 завершена!
